# Notebook 3 — Model Evaluation
**Project:** Evaluating Robustness of Lightweight Object Detection Models Against Adversarial Noise
**Step:** Run YOLOv8n and YOLOv5n inference on clean + noisy images, compute TP/FP/FN, save results to CSV.

In [16]:
import os, sys, json, time, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
from pycocotools.coco import COCO
from tqdm import tqdm
warnings.filterwarnings('ignore')

## 1. Imports & Configuration

In [17]:
# ── 1. Paths ──────────────────────────────────────────────────────────────────
DATA_RAW    = r'D:\Lightweight_Object_Detection_Robustness\data\raw'
ANN_FILE    = DATA_RAW + r'\annotations_trainval2017\annotations\instances_val2017.json'
IMG_DIR     = DATA_RAW + r'\val2017\val2017'
PROC_DIR    = r'D:\Lightweight_Object_Detection_Robustness\data\processed'
RESULTS_DIR = r'D:\Lightweight_Object_Detection_Robustness\results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Settings
SEED           = 42
CONF_THRESH    = 0.25
NMS_IOU_THRESH = 0.45
METRIC_IOU     = 0.50
SUBSET_SIZE    = None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', DEVICE)
print('Subset :', SUBSET_SIZE if SUBSET_SIZE else 'FULL')

Device : cpu
Subset : FULL


## 2. Load Dataset & Noise Functions

In [18]:
# ── 2. Load COCO & Filtered IDs ───────────────────────────────────────────────
coco = COCO(ANN_FILE)

with open(PROC_DIR + r'\filtered_img_ids.json') as f:
    all_img_ids = json.load(f)

if SUBSET_SIZE:
    rng     = np.random.default_rng(SEED)
    img_ids = rng.choice(all_img_ids, size=SUBSET_SIZE, replace=False).tolist()
else:
    img_ids = all_img_ids

print(f'Images to evaluate: {len(img_ids)}')

loading annotations into memory...
Done (t=0.46s)
creating index...
index created!
Images to evaluate: 3028


## 3. Metric Helper Functions

In [19]:
# ── 3. Traffic Classes ────────────────────────────────────────────────────────
TRAFFIC_CAT_IDS = [1, 2, 3, 4, 6, 8, 10, 13]
CAT_ID_TO_NAME  = {1:'person', 2:'bicycle', 3:'car', 4:'motorcycle',
                   6:'bus', 8:'truck', 10:'traffic light', 13:'stop sign'}

# YOLO 0-indexed class IDs (maps to COCO IDs above in same order)
YOLO_CLASS_IDS = [0, 1, 2, 3, 5, 7, 9, 11]
YOLO_TO_COCO   = dict(zip(YOLO_CLASS_IDS, TRAFFIC_CAT_IDS))

## 4. Load Models

In [20]:
# ── 4. Noise Functions ────────────────────────────────────────────────────────
import albumentations as A

GAUSSIAN_SIGMA = {1: 8,    3: 25,   5: 50}
BLUR_KERNEL    = {1: 5,    3: 15,   5: 25}
FOG_BETA       = {1: 0.1,  3: 0.3,  5: 0.6}
RAIN_DROPS     = {1: 500,  3: 1500, 5: 3000}
RAIN_ALPHA     = {1: 0.15, 3: 0.30, 5: 0.50}
np.random.seed(SEED)


def apply_gaussian(img, severity):
    sigma = GAUSSIAN_SIGMA[severity]
    noise = np.random.randn(*img.shape).astype(np.float32) * sigma
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)


def apply_motion_blur(img, severity):
    k      = BLUR_KERNEL[severity]
    kernel = np.zeros((k, k), dtype=np.float32)
    kernel[k // 2, :] = 1.0 / k
    return cv2.filter2D(img, -1, kernel)


def apply_fog(img, severity):
    beta = FOG_BETA[severity]
    transform = A.Compose([A.RandomFog(fog_coef_lower=beta, fog_coef_upper=beta,
                                        alpha_coef=0.1, p=1.0)])
    return transform(image=img)['image']


def apply_rain(img, severity):
    img_f      = img.copy().astype(np.float32)
    n_drops    = RAIN_DROPS[severity]
    alpha      = RAIN_ALPHA[severity]
    h, w       = img.shape[:2]
    rng        = np.random.default_rng(SEED)
    xs         = rng.integers(0, w, n_drops)
    ys         = rng.integers(0, h, n_drops)
    lengths    = rng.integers(10, 30, n_drops)
    rain_layer = np.zeros_like(img_f)
    for x, y, length in zip(xs, ys, lengths):
        x2 = int(np.clip(x + length * np.cos(np.radians(80)), 0, w - 1))
        y2 = int(np.clip(y + length * np.sin(np.radians(80)), 0, h - 1))
        cv2.line(rain_layer, (x, y), (x2, y2), (200, 200, 200), 1)
    return np.clip(img_f * (1 - alpha) + rain_layer * alpha, 0, 255).astype(np.uint8)


NOISE_FNS = {
    'gaussian'    : apply_gaussian,
    'motion_blur' : apply_motion_blur,
    'fog'         : apply_fog,
    'rain'        : apply_rain,
}
print('Noise functions ready')

Noise functions ready


## 5. Inference Functions

In [21]:
# ── 5. Load Models ────────────────────────────────────────────────────────────
from ultralytics import YOLO

print('Loading YOLOv8n ...')
yolov8n = YOLO('yolov8n.pt')
print('YOLOv8n')

print('Loading YOLOv5n ...')
yolov5n = torch.hub.load('ultralytics/yolov5', 'yolov5n', pretrained=True, verbose=False)
yolov5n.conf = CONF_THRESH
yolov5n.iou  = NMS_IOU_THRESH
print('YOLOv5n')

Loading YOLOv8n ...
YOLOv8n
Loading YOLOv5n ...
requirements: Ultralytics requirement ['setuptools>=70.0.0'] not found, attempting AutoUpdate...

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

requirements: AutoUpdate success  1.5s
WARNING requirements: Restart runtime or rerun command for updates to take effect



YOLOv5  2026-3-16 Python-3.10.10 torch-2.10.0+cpu CPU

Fusing layers... 
YOLOv5n summary: 213 layers, 1867405 parameters, 0 gradients, 4.5 GFLOPs
Adding AutoShape... 


YOLOv5n


## 6. Pre-load Ground-Truth Annotations

In [22]:
# ── 6. Metric Helpers ─────────────────────────────────────────────────────────
def iou_box(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter  = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)


def match(pred_boxes, pred_cls, gt_boxes, gt_cls, iou_thresh=0.5):
    tp = fp = 0
    matched = set()
    for pb, pc in zip(pred_boxes, pred_cls):
        best_iou, best_i = 0, -1
        for i, (gb, gc) in enumerate(zip(gt_boxes, gt_cls)):
            if i in matched or pc != gc: continue
            iou = iou_box(pb, gb)
            if iou > best_iou: best_iou, best_i = iou, i
        if best_iou >= iou_thresh and best_i >= 0:
            tp += 1; matched.add(best_i)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched)
    return tp, fp, fn


def metrics(tp, fp, fn):
    p  = tp / (tp + fp + 1e-8)
    r  = tp / (tp + fn + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    return round(p, 4), round(r, 4), round(f1, 4)

## 7. Main Evaluation Loop

In [23]:
# ── 7. Pre-cache Ground Truth ─────────────────────────────────────────────────
gt_cache = {}
for img_id in tqdm(img_ids, desc='Caching GT'):
    ann_ids = coco.getAnnIds(imgIds=img_id, catIds=TRAFFIC_CAT_IDS)
    anns    = coco.loadAnns(ann_ids)
    gt_cache[img_id] = {
        'boxes'  : [[a['bbox'][0], a['bbox'][1],
                     a['bbox'][0]+a['bbox'][2],
                     a['bbox'][1]+a['bbox'][3]] for a in anns],
        'classes': [a['category_id'] for a in anns]
    }
print('GT cached')

Caching GT: 100%|██████████| 3028/3028 [00:00<00:00, 104317.58it/s]

GT cached


In [24]:
# ── 8. Inference Wrappers ─────────────────────────────────────────────────────
def infer_v8(img_rgb):
    results = yolov8n.predict(img_rgb, conf=CONF_THRESH, iou=NMS_IOU_THRESH,
                               classes=YOLO_CLASS_IDS, verbose=False, device=DEVICE)
    boxes, classes = [], []
    for r in results:
        for box in r.boxes:
            cls = int(box.cls.item())
            if cls not in YOLO_TO_COCO: continue
            boxes.append(box.xyxy.cpu().numpy()[0].tolist())
            classes.append(YOLO_TO_COCO[cls])
    return boxes, classes


def infer_v5(img_rgb):
    results = yolov5n(img_rgb, size=640)
    df      = results.pandas().xyxy[0]
    df      = df[df['class'].isin(YOLO_CLASS_IDS) & (df['confidence'] >= CONF_THRESH)]
    boxes   = df[['xmin','ymin','xmax','ymax']].values.tolist()
    classes = [YOLO_TO_COCO[int(c)] for c in df['class'].tolist()]
    return boxes, classes

## 8. Save Results

In [ ]:
# ── 9. Main Evaluation Loop ───────────────────────────────────────────────────
MODEL_MAP = {'yolov8n': infer_v8, 'yolov5n': infer_v5}
SEVERITIES = [1, 3, 5]

all_results = []

for model_name, infer_fn in MODEL_MAP.items():
    # ── clean baseline ──
    for noise_label, noise_fn, severity in \
        [('clean', None, 0)] + \
        [(n, f, s) for n, f in NOISE_FNS.items() for s in SEVERITIES]:

        tp_t = fp_t = fn_t = 0
        per_class = defaultdict(lambda: [0, 0, 0])  # tp, fp, fn

        for img_id in img_ids:
            info     = coco.loadImgs(img_id)[0]
            img_path = os.path.join(IMG_DIR, info['file_name'])
            if not os.path.exists(img_path): continue

            img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            if noise_fn:
                img = noise_fn(img, severity)

            pred_boxes, pred_cls = infer_fn(img)
            gt = gt_cache[img_id]

            tp, fp, fn = match(pred_boxes, pred_cls, gt['boxes'], gt['classes'], METRIC_IOU)
            tp_t += tp; fp_t += fp; fn_t += fn

            for cat_id in TRAFFIC_CAT_IDS:
                pb = [b for b, c in zip(pred_boxes, pred_cls) if c == cat_id]
                gb = [b for b, c in zip(gt['boxes'], gt['classes']) if c == cat_id]
                t, f_p, f_n = match(pb, [cat_id]*len(pb), gb, [cat_id]*len(gb), METRIC_IOU)
                per_class[cat_id][0] += t
                per_class[cat_id][1] += f_p
                per_class[cat_id][2] += f_n

        p, r, f1 = metrics(tp_t, fp_t, fn_t)
        row = {'model': model_name, 'noise': noise_label, 'severity': severity,
               'tp': tp_t, 'fp': fp_t, 'fn': fn_t,
               'precision': p, 'recall': r, 'f1': f1}

        for cat_id in TRAFFIC_CAT_IDS:
            t, fp_c, fn_c = per_class[cat_id]
            _, _, f = metrics(t, fp_c, fn_c)
            row[f'f1_{CAT_ID_TO_NAME[cat_id].replace(" ","_")}'] = f

        all_results.append(row)
        print(f'{model_name:8s} | {noise_label:12s} | s={severity} | '
              f'P={p:.3f}  R={r:.3f}  F1={f1:.3f}')

yolov8n  | clean        | s=0 | P=0.812  R=0.535  F1=0.645
yolov8n  | gaussian     | s=1 | P=0.825  R=0.503  F1=0.625
yolov8n  | gaussian     | s=3 | P=0.817  R=0.372  F1=0.511
yolov8n  | gaussian     | s=5 | P=0.802  R=0.197  F1=0.316
yolov8n  | motion_blur  | s=1 | P=0.809  R=0.466  F1=0.591


## 9. Quick Summary Table

In [15]:
# ── 10. Compute RPD & Save ────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)

for model_name in ['yolov8n', 'yolov5n']:
    clean_f1 = results_df.loc[(results_df['model'] == model_name) &
                               (results_df['noise'] == 'clean'), 'f1'].values[0]
    mask = results_df['model'] == model_name
    results_df.loc[mask, 'clean_f1'] = clean_f1

results_df['rpd'] = ((results_df['clean_f1'] - results_df['f1']) /
                     (results_df['clean_f1'] + 1e-8)).clip(lower=0).round(4)

results_df.to_csv(RESULTS_DIR + r'\evaluation_results.csv', index=False)
print('Saved → results/evaluation_results.csv')
print()
print(results_df[['model','noise','severity','precision','recall','f1','rpd']].to_string(index=False))

Saved → results/evaluation_results.csv

  model       noise  severity  precision  recall     f1    rpd
yolov8n       clean         0     0.8482  0.5828 0.6909 0.0000
yolov8n    gaussian         1     0.8448  0.5556 0.6703 0.0298
yolov8n    gaussian         3     0.8607  0.3923 0.5389 0.2200
yolov8n    gaussian         5     0.8396  0.2018 0.3254 0.5290
yolov8n motion_blur         1     0.8158  0.4921 0.6139 0.1114
yolov8n motion_blur         3     0.7500  0.2653 0.3920 0.4326
yolov8n motion_blur         5     0.6887  0.1655 0.2669 0.6137
yolov8n         fog         1     0.8510  0.5828 0.6918 0.0000
yolov8n         fog         3     0.8448  0.5556 0.6703 0.0298
yolov8n         fog         5     0.8411  0.4921 0.6209 0.1013
yolov8n        rain         1     0.8367  0.4649 0.5977 0.1349
yolov8n        rain         3     0.7838  0.1315 0.2252 0.6740
yolov8n        rain         5     0.5000  0.0023 0.0045 0.9935
yolov5n       clean         0     0.7977  0.6168 0.6957 0.0000
yolov5n    gaus